# 02 · Población vs. comercio (Censo 2020 + DENUE)

**Objetivo:** distinguir zonas **residenciales**, **comerciales** y **mixtas** cruzando la población del Censo 2020 con los establecimientos del DENUE en los mismos hexágonos H3.

**Fuentes (se descargan solas):**
- Censo 2020, resultados por AGEB y manzana urbana de Querétaro (CSV, unos MB)
- Marco Geoestadístico 2020 de Querétaro (62.7 MB), del que solo se usan las manzanas

**Unidad común:** cada manzana se asigna al hexágono que contiene su centroide, y se suma su población. Para comparar con el mismo año, el DENUE que se usa es el de **04/2020**.

## 0. Librerías y rutas

In [ ]:
# Requiere: conda install -n tesis-qro -c conda-forge geopandas contextily
from pathlib import Path
import tempfile, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import h3

ROOT = Path.cwd().parent
RAW, PROC, FIG = ROOT / "data/raw", ROOT / "data/processed", ROOT / "figures"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)
TMP = Path(tempfile.gettempdir()) / "inegi_tmp"
TMP.mkdir(exist_ok=True)

URL_CENSO = "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/ageb_manzana/ageb_mza_urbana_22_cpv2020_csv.zip"
URL_MG = "https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/889463807469/22_queretaro.zip"

MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}
RES = 8
EDICION_DENUE = "2020-04"

## 1. Descarga

El CSV del Censo se guarda en `data/raw/`. El Marco Geoestadístico se baja a una carpeta temporal fuera de Dropbox, y de él solo se guardan los centroides de las manzanas de la zona metropolitana (sección 2).

In [ ]:
def descargar(url, destino, intentos=3):
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return destino
    for i in range(intentos):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as r, open(destino, "wb") as f:
                while bloque := r.read(1 << 20):
                    f.write(bloque)
            print(f"Descargado: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
            return destino
        except Exception as e:
            print(f"  intento {i + 1} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")

zip_censo = descargar(URL_CENSO, RAW / URL_CENSO.split("/")[-1])

## 2. Manzanas: centroides de la zona metropolitana

Del zip del Marco Geoestadístico se extrae solo la capa de manzanas (`22m.shp` y sus archivos hermanos). Se filtran los 4 municipios y se calcula el centroide de cada manzana en la proyección original (metros), para luego pasarlo a latitud y longitud.

El resultado se guarda en `data/processed/manzanas_zmq_2020.parquet`. Si ese archivo ya existe, la sección no vuelve a descargar nada.

In [ ]:
SALIDA_MZA = PROC / "manzanas_zmq_2020.parquet"

if SALIDA_MZA.exists():
    mza = pd.read_parquet(SALIDA_MZA)
    print("Centroides leídos de", SALIDA_MZA.name)
else:
    zip_mg = descargar(URL_MG, TMP / "22_queretaro.zip")
    destino = TMP / "mg_22"
    with zipfile.ZipFile(zip_mg) as z:
        capa = [n for n in z.namelist() if Path(n).stem.lower() == "22m"]
        if not capa:
            raise FileNotFoundError("No encontré la capa 22m (manzanas) en el zip del Marco Geoestadístico")
        z.extractall(destino, members=capa)
    shp = next(destino.rglob("22m.shp"))
    gdf = gpd.read_file(shp)
    gdf.columns = [c.upper() if c != "geometry" else c for c in gdf.columns]
    gdf = gdf[pd.to_numeric(gdf["CVE_MUN"]).isin(MUNICIPIOS_ZMQ)]

    cent = gdf.geometry.centroid                       # en metros (proyección cónica del INEGI)
    cent_ll = gpd.GeoSeries(cent, crs=gdf.crs).to_crs(4326)
    mza = pd.DataFrame({
        "CVEGEO": gdf["CVEGEO"].values,
        "lat": cent_ll.y.values,
        "lon": cent_ll.x.values,
        "area_m2": gdf.geometry.area.values,
    })
    mza.to_parquet(SALIDA_MZA, index=False)
    print(f"Manzanas de la ZM: {len(mza):,} → {SALIDA_MZA.name}")

mza.head()

## 3. Censo 2020 por manzana

El CSV trae renglones de totales (por municipio, localidad y AGEB) que se quitan quedándose con `MZA != 0`. La clave `CVEGEO` se arma con entidad (2) + municipio (3) + localidad (4) + AGEB (4) + manzana (3) dígitos, igual que en el Marco Geoestadístico.

El INEGI oculta con `*` algunos valores de manzanas muy pequeñas (confidencialidad). Esos valores quedan como faltantes y se cuenta cuántos hay.

In [ ]:
with zipfile.ZipFile(zip_censo) as z:
    nombre = max((i for i in z.infolist() if i.filename.lower().endswith(".csv")), key=lambda i: i.file_size)
    censo = pd.read_csv(z.open(nombre), dtype=str, encoding="utf-8", low_memory=False)
censo.columns = censo.columns.str.upper()

censo = censo[pd.to_numeric(censo["MUN"]).isin(MUNICIPIOS_ZMQ) & (pd.to_numeric(censo["MZA"]) != 0)].copy()
censo["CVEGEO"] = (censo["ENTIDAD"].str.zfill(2) + censo["MUN"].str.zfill(3) + censo["LOC"].str.zfill(4)
                   + censo["AGEB"].str.zfill(4) + censo["MZA"].str.zfill(3))

VARS = ["POBTOT", "TVIVHAB"]
for v in VARS:
    censo[v] = pd.to_numeric(censo[v], errors="coerce")       # '*' y 'N/D' → NaN
print(f"Manzanas en el Censo (ZM): {len(censo):,}")
print("Valores ocultos o faltantes:", censo[VARS].isna().sum().to_dict())
print(f"Población total en manzanas urbanas: {censo['POBTOT'].sum():,.0f}")

## 4. Unir Censo y manzanas, y llevar todo a hexágonos

Revisa la tasa de cruce: idealmente casi todas las manzanas del Censo encuentran su polígono.

In [ ]:
mzc = censo[["CVEGEO"] + VARS].merge(mza, on="CVEGEO", how="left", indicator=True)
tasa = (mzc["_merge"] == "both").mean()
print(f"Manzanas del Censo con polígono: {tasa:.1%}")
mzc = mzc[mzc["_merge"] == "both"].drop(columns="_merge")

mzc["h3"] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(mzc["lat"], mzc["lon"])]
pob_hex = mzc.groupby("h3")[VARS].sum(min_count=1)

# DENUE del mismo año
den = pd.read_parquet(PROC / f"denue_zmq_{EDICION_DENUE}.parquet", columns=["latitud", "longitud"])
den["h3"] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(den["latitud"], den["longitud"])]
est_hex = den.groupby("h3").size().rename("establecimientos")

hexes = pob_hex.join(est_hex, how="outer").fillna({"POBTOT": 0, "TVIVHAB": 0, "establecimientos": 0})
hexes = hexes[(hexes["POBTOT"] > 0) | (hexes["establecimientos"] > 0)]
print(f"Hexágonos con población o comercio: {len(hexes):,}")
hexes.describe().round(1)

## 5. Clasificación: residencial, comercial, mixto

Hay dos formas de leer la relación:

- **A. Cuatro tipos de zona (recomendada para presentar):** se compara cada hexágono contra la mediana de población y la mediana de establecimientos.
- **B. Intensidad comercial (sección 7):** establecimientos por cada mil habitantes, en escala continua.

| Población | Comercio | Tipo |
|---|---|---|
| ≥ mediana | < mediana | Residencial |
| < mediana | ≥ mediana | Comercial |
| ≥ mediana | ≥ mediana | Mixto |
| < mediana | < mediana | Baja densidad |

In [ ]:
p50 = hexes.loc[hexes["POBTOT"] > 0, "POBTOT"].median()
e50 = hexes.loc[hexes["establecimientos"] > 0, "establecimientos"].median()
alta_pob, alto_com = hexes["POBTOT"] >= p50, hexes["establecimientos"] >= e50

hexes["tipo"] = np.select(
    [alta_pob & ~alto_com, ~alta_pob & alto_com, alta_pob & alto_com],
    ["Residencial", "Comercial", "Mixto"], default="Baja densidad")
print(f"Medianas → población: {p50:,.0f} hab · establecimientos: {e50:,.0f}")
resumen = hexes.groupby("tipo").agg(hexagonos=("tipo", "size"), poblacion=("POBTOT", "sum"),
                                    establecimientos=("establecimientos", "sum"))
resumen["% población"] = (resumen["poblacion"] / resumen["poblacion"].sum() * 100).round(1)
resumen["% comercio"] = (resumen["establecimientos"] / resumen["establecimientos"].sum() * 100).round(1)
resumen

## 6. Mapa A · Tipos de zona

In [ ]:
from matplotlib.collections import PolyCollection
from matplotlib.patches import Patch

COLORES = {"Residencial": "#2E6F95", "Comercial": "#E8822F", "Mixto": "#14213D", "Baja densidad": "#D9D3C7"}
REFERENCIAS = {
    "Centro histórico": (20.593, -100.392), "Juriquilla": (20.710, -100.450),
    "El Pueblito\n(Corregidora)": (20.539, -100.444), "La Cañada\n(El Marqués)": (20.613, -100.326),
    "Huimilpan": (20.372, -100.276),
}

def base_mapa(cells, figsize=(10, 11)):
    polys = [[(lng, lat) for lat, lng in h3.cell_to_boundary(c)] for c in cells]
    xs = [x for p in polys for x, _ in p]; ys = [y for p in polys for _, y in p]
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(min(xs) - 0.01, max(xs) + 0.01); ax.set_ylim(min(ys) - 0.01, max(ys) + 0.01)
    ax.set_aspect(1 / np.cos(np.radians(20.6))); ax.set_axis_off()
    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.Esri.WorldGrayCanvas, zoom=12, attribution_size=7)
    except Exception as e:
        print("Sin mapa base:", e)
    return fig, ax, polys

def referencias(ax):
    for nombre, (la, lo) in REFERENCIAS.items():
        ax.plot(lo, la, "o", ms=4, color="#14213D")
        ax.annotate(nombre, (lo, la), xytext=(6, 6), textcoords="offset points", fontsize=10,
                    color="#14213D", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))

fig, ax, polys = base_mapa(hexes.index)
ax.add_collection(PolyCollection(polys, facecolors=hexes["tipo"].map(COLORES).values,
                                 edgecolor="white", linewidth=0.3, alpha=0.85))
referencias(ax)
ax.legend(handles=[Patch(color=c, label=f"{t} ({resumen.loc[t, 'hexagonos']:,} hex.)")
                   for t, c in COLORES.items() if t in resumen.index],
          loc="lower left", frameon=True, fontsize=11, title="Tipo de zona")
ax.set_title(f"Zonas residenciales, comerciales y mixtas · Censo 2020 + DENUE {EDICION_DENUE}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "04_tipos_de_zona.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Mapa B · Establecimientos por cada mil habitantes

Solo para hexágonos con al menos 100 habitantes; con menos, la razón se dispara y no dice nada.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

con_pob = hexes[hexes["POBTOT"] >= 100].copy()
con_pob["est_x_mil"] = con_pob["establecimientos"] / con_pob["POBTOT"] * 1000

BORDES = [0, 10, 20, 40, 80, 160, max(161, con_pob["est_x_mil"].max() + 1)]
ETIQ = ["0–9", "10–19", "20–39", "40–79", "80–159", "160 o más"]
cmap = ListedColormap(["#EAF1F6", "#BFD6E6", "#8DB7D3", "#E8A25F", "#C2410C", "#7F2704"])
norm = BoundaryNorm(BORDES, cmap.N)

fig, ax, polys = base_mapa(con_pob.index)
coll = PolyCollection(polys, array=con_pob["est_x_mil"].values, cmap=cmap, norm=norm,
                      edgecolor="white", linewidth=0.3, alpha=0.85)
ax.add_collection(coll); referencias(ax)
cbar = fig.colorbar(coll, ax=ax, shrink=0.5, ticks=[(a + b) / 2 for a, b in zip(BORDES[:-1], BORDES[1:])])
cbar.ax.set_yticklabels(ETIQ); cbar.set_label("Establecimientos por cada mil habitantes")
ax.set_title(f"Intensidad comercial · Censo 2020 + DENUE {EDICION_DENUE}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "05_establecimientos_por_mil_hab.png", dpi=200, bbox_inches="tight"); plt.show()

print(f"Mediana de la ZM: {con_pob['est_x_mil'].median():.1f} establecimientos por mil habitantes")

## 8. Guardar la tabla por hexágono (para los modelos)

In [ ]:
hexes.reset_index().rename(columns={"index": "h3"}).to_parquet(PROC / f"hex_res{RES}_pob_comercio_2020.parquet", index=False)
print("Guardado:", f"hex_res{RES}_pob_comercio_2020.parquet")